# Create Agents to Research and Write an Article with CrewAi

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29 crewai[litellm] crewai[anthropic]
```

In [1]:
# Warning control:
import warnings
import os

warnings.filterwarnings('ignore')

# Disable OpenTelemetry SDK to prevent automatic tracing initialization
#os.environ["OTEL_SDK_DISABLED"] = "true"

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("BASE_URL")
model_name = "gpt-4.1-mini"

llm = ChatOpenAI(
    api_key=api_key,
    base_url=base_url,
    model=model_name,
    temperature=0.7
)

#os.environ["OPENAI_BASE_URL"] = "http://localhost:11434"
#os.environ["MODEL_NAME"] = "ollama/gpt-oss:20b" 

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [4]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and accurate content about {topic}",
    backstory="You are planning a blog article "
              "about the topic: {topic}. "
              "You gather information that helps the audience "
              "learn something new "
              "and make informed decisions. "
              "Your work forms the foundation for the content writer "
              "to write an article about this topic.",
    allow_delegation=False,
    verbose=True, llm=llm
)


### Agent: Writer

In [5]:
writer = Agent(
    role="Content Writer",
    goal="Write an engaging and accurate opinion article "
         "about the topic: {topic}",
    backstory="You are writing a new opinion article "
              "about the topic: {topic}. "
              "Your writing is based on the work of the content planner, "
              "who provides an outline and relevant background "
              "about the topic. "
              "You follow the main objectives and direction of the outline "
              "provided by the content planner. "
              "You also provide objective and unbiased viewpoints "
              "and support them with the information provided "
              "by the content planner. "
              "In your opinion article, you clearly indicate "
              "which statements are personal opinions "
              "and which are objective and factual statements.",
    allow_delegation=False,
    verbose=True, llm=llm
)


### Agent: Editor

In [6]:
editor = Agent(
    role="Editor",
    goal="Edit the received blog article "
         "and align it with the organization's writing style.",
    backstory="You are an editor who receives the blog article "
              "from the content writer. "
              "Your goal is to review the blog article "
              "to ensure it follows journalistic best practices, "
              "presents balanced and fair perspectives "
              "when expressing opinions or claims, "
              "and avoids controversial topics "
              "or opinions as much as possible.",
    allow_delegation=False,
    verbose=True, llm=llm
)


## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [7]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players "
            "and important news about {topic}.\n"
        "2. Identify the target audience, "
            "considering their interests and pain points.\n"
        "3. Prepare a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or resources."
    ),
    expected_output="A comprehensive content planning document "
        "including an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)


### Task: Write

In [8]:
write = Task(
    description=(
        "1. Use the content plan to write an engaging blog article "
            "about {topic}.\n"
        "2. Incorporate SEO keywords naturally into the text.\n"
        "3. Make the sections and subheadings engaging "
            "and appropriately named.\n"
        "4. Ensure the article has a proper structure: "
            "an engaging introduction, a substantial main body, "
            "and a concise conclusion.\n"
        "5. Review the text for grammatical errors "
            "and consistency with the brand tone.\n"
    ),
    expected_output="A well-written blog article in English "
        "in Markdown format, ready for publication; "
        "each section should contain 2 or 3 paragraphs.",
    agent=writer,
)


### Task: Edit

In [9]:
edit = Task(
    description=("Review and edit the provided blog article for "
                 "grammatical errors and "
                 "consistency with the brand tone."),
    expected_output="A well-written blog article in English in Markdown format, "
                    "ready for publication; "
                    "each section should contain 2 or 3 paragraphs.",
    agent=editor
)


## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=True` allows you to see all the logs of the execution. 

In [10]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=True
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [ ]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

2026-08-13 17:05:47,058 - 74960 - manager.py-manager:282 - WARNING: Error in TokenCalcHandler.on_llm_start callback: KeyError('Could not automatically map gpt-4.1-mini to a tokeniser. Please use `tiktoken.get_encoding` to explicitly get the tokeniser you expect.')
2026-08-13 17:05:47,058 - 74960 - manager.py-manager:282 - WARNING: Error in TokenCalcHandler.on_llm_start callback: KeyError('Could not automatically map gpt-4.1-mini to a tokeniser. Please use `tiktoken.get_encoding` to explicitly get the tokeniser you expect.')
2026-08-13 17:05:47,059 - 74960 - manager.py-manager:282 - WARNING: Error in TokenCalcHandler.on_llm_start callback: KeyError('Could not automatically map gpt-4.1-mini to a tokeniser. Please use `tiktoken.get_encoding` to explicitly get the tokeniser you expect.')
2026-08-13 17:05:47,059 - 74960 - manager.py-manager:282 - WARNING: Error in TokenCalcHandler.on_llm_start callback: KeyError('Could not automatically map gpt-4.1-mini to a tokeniser. Please use `tiktoken.

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players and important news about Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Prepare a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or resources.


> Entering new CrewAgentExecutor chain...


2026-08-13 17:05:49,499 - 73900 - __init__.py-__init__:247 - WARNING: Transient error HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by SSLError(SSLError(1, '[SSL: TLSV1_ALERT_ACCESS_DENIED] tlsv1 alert access denied (_ssl.c:1007)'))) encountered while exporting span batch, retrying in 1.02s.
2026-08-13 17:05:51,014 - 73900 - __init__.py-__init__:247 - WARNING: Transient error HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by SSLError(SSLError(1, '[SSL: TLSV1_ALERT_ACCESS_DENIED] tlsv1 alert access denied (_ssl.c:1007)'))) encountered while exporting span batch, retrying in 1.78s.
2026-08-13 17:05:53,271 - 73900 - __init__.py-__init__:247 - WARNING: Transient error HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by SSLError(SSLError(1, '[SSL: TLSV1_ALERT_ACCESS_DENIED] tlsv1 alert access denied (_s

Thought: I now can give a great answer

Final Answer:

---

**Comprehensive Content Planning Document for Blog Article: "Artificial Intelligence"**

---

### 1. Prioritizing Latest Trends, Key Players, and Important News in Artificial Intelligence (AI)

**Latest Trends:**
- **Generative AI**: AI models like GPT-4, DALL·E, and Stable Diffusion revolutionizing content creation (text, images, code).
- **AI in Healthcare**: AI-driven diagnostics, personalized medicine, and drug discovery advancements.
- **AI Ethics and Regulation**: Increasing focus on transparency, bias reduction, and regulatory frameworks worldwide.
- **AI and Automation**: Expansion of AI-powered automation in industries such as manufacturing, customer service (chatbots), and finance.
- **Edge AI**: AI processing on devices locally (IoT, smartphones) to reduce latency and enhance privacy.
- **Multimodal AI**: Models combining text, images, and other data types for richer understanding and interaction.
- **AI in Climate 

- Display the results of your execution as markdown in the notebook.

In [75]:
from IPython.display import HTML
import markdown

html_content = markdown.markdown(result.raw)

HTML(f'''
<style>
    .rtl-content * {{
        direction: rtl !important;
        text-align: right !important;
    }}
</style>
<div class="rtl-content" style="
    direction: rtl;
    text-align: right; 
    font-family: Tahoma, Arial, sans-serif; 
    font-size: 15px; 
    line-height: 2;
    padding: 20px;
">
    {html_content}
</div>
''')


<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).